# 05 — Fit Predictor (supervised, optional)
Build (resume, job) features → classify shortlisting. Evaluate precision/recall/ROC-AUC.


## Objective

The objective of this notebook is to estimate how well a candidate's resume matches a specific job posting.

The workflow includes:

- Loading the trained TF-IDF vectorizer
- Transforming resumes and job descriptions into numerical vectors
- Computing cosine similarity
- Calculating a job fit percentage
- Interpreting the similarity score

The fit score helps candidates understand how closely their resumes align with job requirements.

In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

import joblib

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# ============================================================
# Load Dataset and Trained Objects
# ============================================================

PROJECT_ROOT = Path("..")

MODEL_DIR = PROJECT_ROOT / "models"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

jobs_df = pd.read_csv(PROCESSED_DATA / "jobs_clean.csv")

tfidf = joblib.load(MODEL_DIR / "job_tfidf_vectorizer.pkl")

print("Dataset and TF-IDF vectorizer loaded successfully!")

Dataset and TF-IDF vectorizer loaded successfully!


In [3]:
# ============================================================
# Create Job Feature Matrix
# ============================================================

job_vectors = tfidf.transform(
    jobs_df["text"]
)

print(job_vectors.shape)

(136759, 5000)


In [4]:
# ============================================================
# Job Fit Score Function
# ============================================================

def calculate_fit_score(resume_text, job_text):
    """
    Calculate the similarity between a resume and a job description.
    """

    # Transform resume
    resume_vector = tfidf.transform([resume_text])

    # Transform job description
    job_vector = tfidf.transform([job_text])

    # Calculate cosine similarity
    similarity = cosine_similarity(
        resume_vector,
        job_vector
    )[0][0]

    # Convert to percentage
    fit_score = similarity * 100

    return round(fit_score, 2)

In [7]:
# ============================================================
# Test the Fit Score
# ============================================================

sample_resume = """
Python
Machine Learning
TensorFlow
Pandas
NumPy
SQL
Scikit-Learn
Deep Learning
"""

# Find the most similar job
resume_vector = tfidf.transform([sample_resume])

similarity_scores = cosine_similarity(
    resume_vector,
    job_vectors
).flatten()

best_index = similarity_scores.argmax()

sample_job = jobs_df.iloc[best_index]["text"]

score = calculate_fit_score(
    sample_resume,
    sample_job
)

print(f"Job Fit Score: {score}%")

Job Fit Score: 59.78%


In [10]:
# ============================================================
# Interpret Fit Score
# ============================================================

def interpret_fit(score):

    if score >= 70:
        return "Excellent Match"

    elif score >= 50:
        return "Good Match"

    elif score >= 30:
        return "Moderate Match"

    else:
        return "Low Match"

print(f"Fit Level: {interpret_fit(score)}")

Fit Level: Good Match


# Conclusion

The Job Fit Predictor successfully estimates how closely a resume matches a job posting.

### Results

- Loaded the trained TF-IDF vectorizer.
- Converted resumes and job descriptions into TF-IDF vectors.
- Calculated cosine similarity between the two vectors.
- Converted similarity into a percentage-based fit score.
- Added qualitative interpretations of the score.

The Job Fit Predictor complements the recommendation system by providing an intuitive measure of resume-job compatibility.